# coaxial 63100 - the gate drivers
Checked in with the stand-in's outputs; `SIMULATED = True                    # False at the bench` and a
port at the bench.
**This arms a power stage.** The 2EDL8034 has no interlock; TIM1's dead
time is all there is between the two FETs of a leg.

Two lifecycles, nested: the stage is `check()`, `arm()`, a duty, `disarm()`,
and the acquisition inside it is the `daq_session` one. The stage's is the
one that has to be asked for by name - `arm()` is the only thing that sets
MOE, and a duty write is refused until it has been called.

In [1]:
import os
import sys
import time

from pathlib import Path

root = Path.cwd()
while not (root / 'host' / 'coaxial').is_dir():
    root = root.parent
sys.path.insert(0, str(root / 'host'))

from coaxial import Coaxial63100, scaling

SIMULATED = True                    # False at the bench
DUTY = 0.25

device = Coaxial63100(port='COM4', simulated_device=SIMULATED, power_afe=False)
stage = device.gates             # the arming policy lives here
daq = device.daq                 # and the acquisition front door
stage.open()
print(device)
print(stage)

<Coaxial63100 Simulated SIMULATED>
<GateStage - check(), arm(), disarm(). A duty write is refused until arm() has been called>


## Dead time, before anything else
Refuses at zero. Read from BDTR every time, not remembered.

In [2]:
state = stage.check()
print('DTG %d, period %d ticks' % (state['deadtime'], state['period']))

DTG 19, period 2376 ticks


## Which half of the board you get
AFE_ON powers the ADC reference. On this bench board the same pin, gated
the other way, is what gives the drivers supply - so with the drivers
powered the board refuses to convert, and there are no currents.

In [3]:
device.write(digital={'AFE_ON': True})            # currents real, no drive
time.sleep(0.3)
daq.configure(accumulate=8, digital=False)
daq.start()

<Coaxial63100 Simulated SIMULATED>

## Arm, then set a duty
Arming is a call. A duty write is refused until it has been made.

In [4]:
# ignore_interlock because this bench board is not modified: the schematic
# wants Cinj and Clevel both above 3 V first and they read 0.79 and 0.08.
stage.arm(bypass_sto=True, ignore_interlock=True)
print('armed:', stage.armed())
device.write(analog={'Phase U': DUTY, 'Phase V': DUTY, 'Phase W': DUTY})

armed: True


{'Phase U': 0.2496842105263158,
 'Phase V': 0.2496842105263158,
 'Phase W': 0.2496842105263158}

## The six gate signals, as one instant
One IDR read on the board. Six asks would be six instants.

In [5]:
snap = device.gate_drivers.state()
print('CNT %d of %d' % (snap['pins_at'], snap['period'] - 1))
for leg in ('U', 'V', 'W'):
    high, low = snap['pins'][leg + 'H'], snap['pins'][leg + 'L']
    print('  %s  high %d  low %d%s'
          % (leg, high, low, '   BOTH ON' if high and low else ''))

CNT 1326 of 2375
  U  high 0  low 1
  V  high 0  low 1
  W  high 0  low 1


## Current, DC link and the two supply senses, with ripple
The live accumulator carries a count, a lowest and a highest per channel,
so ripple is measured rather than inferred from one sample.

The phase readings here are OFFSET, not current: nothing is armed and no
current flows, so what they show is where each channel sits with zero
through the shunt. Zeroing them is `calibration.zero(index)`, against an
instrument - invariant 7. The ripple beside them is real either way.

In [6]:
live = daq.latest()
params = device.analog.scaling()      # the board's record, not this file's
units = {f['signal']: (f['unit'], f['differential'])
         for f in daq.layout['fields']}
for name in ('Phase U', 'Phase V', 'Phase W', 'DC bus', '+5V', 'Vgate'):
    unit, diff = units[name]
    to = scaling.converter(unit, diff, signal=name,   # three mV channels,
                           params=params)             # three dividers
    print('%-8s %+9.3f %-2s  p-p %7.3f  over %d'
          % (name, to(live['mean'][name]), scaling.symbol(unit, name),
             abs(to(live['highest'][name]) - to(live['lowest'][name])),
             live['count'][name]))

Phase U     +8.875 A   p-p   0.760  over 26
Phase V    -50.806 A   p-p   0.760  over 26
Phase W     +2.265 A   p-p   0.760  over 27
DC bus     +24.775 V   p-p   0.143  over 26
+5V         +5.106 V   p-p   0.012  over 27
Vgate       +0.345 V   p-p   0.040  over 27


## A timed burst
Unlimited rate, finite record count - the board allows that combination
because the run ends. Start and stop are a round trip each, about 15 ms,
so an ask under that is bounded by the link and the stamps say so.

Through the front door, `records=` and `interval_us=` included - they are
passed through to the raw ops, so the burst keeps the door's layout
handling and the sample counting instead of re-deriving either. The raw
vocabulary itself still lives at `device.board.daq`.

In [7]:
daq.stop()
daq.configure([f['signal'] for f in daq.layout['fields']],
              digital=False, accumulate=1, records=512, interval_us=0)
daq.start()
time.sleep(0.100)
daq.stop()

got = []                     # after stop() the raw op drains the ring;
while True:                  # read(-1) needs the reader start() put on the
    batch = daq.acquire()    # link, and stop() has taken it down
    if not batch:
        break
    got.extend(batch)

span = (got[-1]['at'] - got[0]['at']) / 475e6 if len(got) > 1 else 0.0
print('%d records over %.3f ms, %.0f us apart, %d dropped'
      % (len(got), span * 1e3, span * 1e6 / max(1, len(got) - 1),
         daq.state()['dropped']))

487 records over 2116.932 ms, 4356 us apart, 0 dropped


In [8]:
stage.disarm()
device.close()